In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs detected:", gpus)

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print("Module 3 GPU ready." if gpus else "No GPU detected.")

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive")

print("\nFolders in My Drive:")
for folder in sorted(drive_root.iterdir()):
    if folder.is_dir():
        print(folder.name)

In [ ]:
from pathlib import Path
import random
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_DIR = Path("/content/drive/MyDrive/Breast_Cancer_Project")
MODULE3_DIR = PROJECT_DIR / "module3_mammogram"

FOLDERS = {
    "data": MODULE3_DIR / "data",
    "models": MODULE3_DIR / "models",
    "results": MODULE3_DIR / "results",
}

for name, folder in FOLDERS.items():
    folder.mkdir(parents=True, exist_ok=True)
    print(f"{name}: {folder}")

print("\nModule 3 folders ready.")

In [ ]:
%pip -q install kagglehub

In [ ]:
import kagglehub
from pathlib import Path

download_dir = Path(
    kagglehub.dataset_download("kmader/mias-mammography")
)

print("Downloaded to:", download_dir)

files = sorted(p for p in download_dir.rglob("*") if p.is_file())

print("Total files:", len(files))
print("\nFirst 20 files:")
for file in files[:20]:
    print(file.relative_to(download_dir))

print("\nAnnotation and archive candidates:")
for file in files:
    if (
        "readme" in file.name.lower()
        or file.suffix.lower() in {".txt", ".csv", ".zip", ".gz", ".tar"}
    ):
        print(file.relative_to(download_dir))

In [ ]:
import shutil

SOURCE_DIR = download_dir / "all-mias"
MIAS_DIR = FOLDERS["data"] / "mini_mias"

shutil.copytree(SOURCE_DIR, MIAS_DIR, dirs_exist_ok=True)

image_files = sorted(MIAS_DIR.glob("mdb*.pgm"))

print("Dataset saved to:", MIAS_DIR)
print("Mammogram images:", len(image_files))

if len(image_files) != 322:
    raise ValueError(f"Expected 322 images, found {len(image_files)}.")

for filename in ["Info.txt", "README"]:
    path = MIAS_DIR / filename

    if path.exists():
        print(f"\n--- {filename}: first 60 lines ---")
        lines = path.read_text(errors="replace").splitlines()
        print("\n".join(lines[:60]))

In [ ]:
annotation_path = MIAS_DIR / "Info.txt"
lines = annotation_path.read_text(errors="replace").splitlines()

print("ANNOTATION INSTRUCTIONS:")
start = next(
    (i for i, line in enumerate(lines) if "1st column" in line),
    0
)

print("\n".join(lines[start:start + 65]))

annotation_lines = [
    line.strip()
    for line in lines
    if line.strip().startswith("mdb")
]

print("\nNumber of annotation rows:", len(annotation_lines))

print("\nFirst 20 annotation rows:")
print("\n".join(annotation_lines[:20]))

print("\nLICENCE:")
print(
    (MIAS_DIR / "Licence.txt").read_text(errors="replace")
)

In [ ]:
import pandas as pd
from pathlib import Path

records = []

for line in annotation_lines:
    # Separate optional notes from the annotation fields.
    fields, marker, note_text = line.partition("*")
    parts = fields.split()
    note = (marker + note_text).strip() or None

    if len(parts) not in {3, 4, 7}:
        raise ValueError(f"Unexpected annotation format: {line}")

    image_id, tissue, abnormality = parts[:3]
    image_number = int(image_id.replace("mdb", ""))

    severity = parts[3] if len(parts) >= 4 else None
    has_location = len(parts) == 7

    x, y, radius = (
        tuple(map(int, parts[4:7]))
        if has_location
        else (None, None, None)
    )

    if has_location:
        if not (0 <= x < 1024 and 0 <= y < 1024 and radius > 0):
            raise ValueError(f"Invalid coordinates: {line}")

    records.append({
        "image_id": image_id,
        "patient_id": (image_number + 1) // 2,
        "tissue": tissue,
        "abnormality": abnormality,
        "severity": severity,
        "is_abnormal": abnormality != "NORM",
        "has_location": has_location,
        "x": x,
        "y_bottom": y,
        "y_top": 1023 - y if has_location else None,
        "radius": radius,
        "annotation_note": note,
        "image_path": str(MIAS_DIR / f"{image_id}.pgm"),
    })

annotations = pd.DataFrame(records)

for column in ["x", "y_bottom", "y_top", "radius"]:
    annotations[column] = annotations[column].astype("Int64")

missing_images = [
    path for path in annotations["image_path"].unique()
    if not Path(path).is_file()
]

if missing_images:
    raise FileNotFoundError(missing_images)

annotations.to_csv(
    FOLDERS["results"] / "mias_annotations.csv",
    index=False
)

print("Annotation rows:", len(annotations))
print("Unique images:", annotations["image_id"].nunique())
print("Unique patients:", annotations["patient_id"].nunique())
print("Lesion rows with coordinates:", annotations["has_location"].sum())

print("\nAbnormal rows without coordinates:")
display(
    annotations.loc[
        annotations["is_abnormal"] & ~annotations["has_location"],
        ["image_id", "abnormality", "severity", "annotation_note"]
    ]
)

print("\nFirst 10 rows:")
display(annotations.head(10))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from PIL import Image
import numpy as np

sample_ids = ["mdb001", "mdb005", "mdb003"]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, image_id in zip(axes, sample_ids):
    image = np.array(Image.open(MIAS_DIR / f"{image_id}.pgm"))

    if image.shape != (1024, 1024):
        raise ValueError(f"Unexpected image shape: {image_id}: {image.shape}")

    rows = annotations[annotations["image_id"] == image_id]

    ax.imshow(image, cmap="gray", origin="upper")

    for _, row in rows[rows["has_location"]].iterrows():
        circle = Circle(
            (int(row["x"]), int(row["y_top"])),
            int(row["radius"]),
            edgecolor="yellow",
            facecolor="none",
            linewidth=2,
        )
        ax.add_patch(circle)

    label = "Abnormal" if rows["is_abnormal"].any() else "Normal"
    ax.set_title(f"{image_id} — {label}")
    ax.axis("off")

plt.tight_layout()

figure_path = FOLDERS["results"] / "annotation_check.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

print("Yellow circles = dataset annotations, not model predictions.")
print("Saved:", figure_path)

In [ ]:
image_table = (
    annotations.groupby("image_id", sort=True)
    .agg(
        patient_id=("patient_id", "first"),
        tissue=("tissue", "first"),
        image_path=("image_path", "first"),
        is_abnormal=("is_abnormal", "max"),
        has_location=("has_location", "max"),
        located_lesion_count=("has_location", "sum"),
    )
    .reset_index()
)

# An abnormal image with any unlocated lesion has incomplete
# localization annotations.
unlocated_ids = set(
    annotations.loc[
        annotations["is_abnormal"] & ~annotations["has_location"],
        "image_id"
    ]
)

image_table["localization_eligible"] = (
    ~image_table["image_id"].isin(unlocated_ids)
)

image_table["label"] = image_table["is_abnormal"].astype(int)

assert len(image_table) == 322
assert image_table["patient_id"].nunique() == 161
assert image_table.groupby("patient_id").size().eq(2).all()

image_table.to_csv(
    FOLDERS["results"] / "mias_image_table.csv",
    index=False
)

print("Total images:", len(image_table))
print("Normal images:", (image_table["label"] == 0).sum())
print("Abnormal images:", (image_table["label"] == 1).sum())
print(
    "Images eligible for localization:",
    image_table["localization_eligible"].sum()
)
print(
    "Images excluded from localization:",
    (~image_table["localization_eligible"]).sum()
)

display(image_table.head(10))

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

# Stratify patients by whether either mammogram is abnormal.
patient_table = (
    image_table.groupby("patient_id")["label"]
    .max()
    .reset_index()
)

train_patients, remaining_patients = train_test_split(
    patient_table,
    test_size=0.30,
    random_state=SEED,
    stratify=patient_table["label"],
)

val_patients, test_patients = train_test_split(
    remaining_patients,
    test_size=0.50,
    random_state=SEED,
    stratify=remaining_patients["label"],
)

patient_sets = {
    "train": set(train_patients["patient_id"]),
    "validation": set(val_patients["patient_id"]),
    "test": set(test_patients["patient_id"]),
}

assert patient_sets["train"].isdisjoint(patient_sets["validation"])
assert patient_sets["train"].isdisjoint(patient_sets["test"])
assert patient_sets["validation"].isdisjoint(patient_sets["test"])

split_lookup = {
    patient_id: split
    for split, patient_ids in patient_sets.items()
    for patient_id in patient_ids
}

image_table["split"] = image_table["patient_id"].map(split_lookup)

assert image_table["split"].notna().all()
assert image_table.groupby("patient_id")["split"].nunique().eq(1).all()

image_table.to_csv(
    FOLDERS["results"] / "mias_image_table.csv",
    index=False
)

summary = (
    image_table.groupby("split")
    .agg(
        patients=("patient_id", "nunique"),
        images=("image_id", "count"),
        abnormal_images=("label", "sum"),
        localization_eligible=("localization_eligible", "sum"),
    )
    .reindex(["train", "validation", "test"])
)

summary["normal_images"] = (
    summary["images"] - summary["abnormal_images"]
)

display(summary)
print("Patient overlap: 0")
print("Split assignments saved.")

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

IMAGE_SIZE = 256

def load_image_and_mask(image_id):
    rows = annotations[annotations["image_id"] == image_id]

    if rows.empty:
        raise ValueError(f"No annotations for {image_id}")

    if (rows["is_abnormal"] & ~rows["has_location"]).any():
        raise ValueError(f"Incomplete localization annotations: {image_id}")

    with Image.open(MIAS_DIR / f"{image_id}.pgm") as img:
        image = img.convert("L")

        if image.size != (1024, 1024):
            raise ValueError(f"Unexpected image size: {image.size}")

        resized_image = image.resize(
            (IMAGE_SIZE, IMAGE_SIZE),
            Image.Resampling.BILINEAR
        )

    # Combine every annotated lesion into one full-resolution mask.
    yy, xx = np.ogrid[:1024, :1024]
    mask = np.zeros((1024, 1024), dtype=bool)

    for _, row in rows[rows["has_location"]].iterrows():
        x = int(row["x"])
        y = int(row["y_top"])
        radius = int(row["radius"])

        mask |= (xx - x)**2 + (yy - y)**2 <= radius**2

    resized_mask = Image.fromarray(
        mask.astype(np.uint8) * 255
    ).resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        Image.Resampling.NEAREST
    )

    image_array = np.asarray(resized_image, dtype=np.float32) / 255.0
    mask_array = (np.asarray(resized_mask) > 0).astype(np.float32)

    return image_array[..., None], mask_array[..., None]


# Inspect training examples only.
preview_ids = (
    image_table.loc[
        (image_table["split"] == "train")
        & image_table["localization_eligible"]
        & image_table["is_abnormal"],
        "image_id"
    ]
    .head(3)
    .tolist()
)

fig, axes = plt.subplots(3, 2, figsize=(8, 12))

for i, image_id in enumerate(preview_ids):
    image, mask = load_image_and_mask(image_id)

    axes[i, 0].imshow(image[..., 0], cmap="gray")
    axes[i, 0].set_title(f"{image_id}: resized mammogram")

    axes[i, 1].imshow(image[..., 0], cmap="gray")
    axes[i, 1].imshow(
        np.ma.masked_where(mask[..., 0] == 0, mask[..., 0]),
        cmap="autumn",
        alpha=0.4,
        vmin=0,
        vmax=1
    )
    axes[i, 1].set_title("Annotation mask overlay")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)
print("Mask values:", np.unique(mask))

In [ ]:
def prepare_split(split_name):
    rows = image_table.loc[
        (image_table["split"] == split_name)
        & image_table["localization_eligible"]
    ].copy()

    images, masks = [], []

    for image_id in rows["image_id"]:
        image, mask = load_image_and_mask(image_id)
        images.append(image)
        masks.append(mask)

    X = np.stack(images).astype(np.float32)
    Y = np.stack(masks).astype(np.float32)

    assert X.shape == Y.shape
    assert np.isfinite(X).all()
    assert np.isin(Y, [0, 1]).all()

    # Every eligible abnormal image must retain a mask after resizing.
    has_mask = Y.sum(axis=(1, 2, 3)) > 0
    assert np.array_equal(
        has_mask,
        rows["is_abnormal"].to_numpy(dtype=bool)
    ), "An abnormal image lost its mask during resizing."

    return X, Y, rows.reset_index(drop=True)


X_train, Y_train, train_rows = prepare_split("train")
X_val, Y_val, val_rows = prepare_split("validation")

print("Training images:", X_train.shape)
print("Training masks:", Y_train.shape)
print("Validation images:", X_val.shape)
print("Validation masks:", Y_val.shape)

print("\nTraining images with lesion masks:",
      int((Y_train.sum(axis=(1, 2, 3)) > 0).sum()))
print("Validation images with lesion masks:",
      int((Y_val.sum(axis=(1, 2, 3)) > 0).sum()))

print(f"\nTraining foreground pixels: {Y_train.mean():.2%}")
print("Image range:", float(X_train.min()), "to", float(X_train.max()))
print("Mask values:", np.unique(Y_train))
print("\nArrays ready. Test data remains reserved.")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

tf.keras.utils.set_random_seed(SEED)

@tf.keras.utils.register_keras_serializable(package="Mammogram")
def localization_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Cross-entropy penalizes incorrect foreground/background scores.
    bce = tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(y_true, y_pred)
    )

    # Compute Dice per image, using only images with lesion masks.
    axes = (1, 2, 3)
    intersection = tf.reduce_sum(y_true * y_pred, axis=axes)
    true_area = tf.reduce_sum(y_true, axis=axes)
    predicted_area = tf.reduce_sum(y_pred, axis=axes)

    dice = (
        (2.0 * intersection + 1e-6)
        / (true_area + predicted_area + 1e-6)
    )

    positive_images = tf.cast(true_area > 0, tf.float32)
    dice_loss = tf.math.divide_no_nan(
        tf.reduce_sum((1.0 - dice) * positive_images),
        tf.reduce_sum(positive_images)
    )

    return bce + dice_loss


def conv_block(x, filters):
    x = layers.Conv2D(
        filters, 3, padding="same",
        activation="relu", kernel_initializer="he_normal"
    )(x)
    x = layers.Conv2D(
        filters, 3, padding="same",
        activation="relu", kernel_initializer="he_normal"
    )(x)
    return x


inputs = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1))

c1 = conv_block(inputs, 16)
c2 = conv_block(layers.MaxPooling2D()(c1), 32)
c3 = conv_block(layers.MaxPooling2D()(c2), 64)

bottleneck = conv_block(layers.MaxPooling2D()(c3), 128)
bottleneck = layers.Dropout(0.3)(bottleneck)

x = layers.UpSampling2D()(bottleneck)
x = conv_block(layers.Concatenate()([x, c3]), 64)

x = layers.UpSampling2D()(x)
x = conv_block(layers.Concatenate()([x, c2]), 32)

x = layers.UpSampling2D()(x)
x = conv_block(layers.Concatenate()([x, c1]), 16)

outputs = layers.Conv2D(
    1, 1, activation="sigmoid", name="localization_scores"
)(x)

model = Model(inputs, outputs, name="mammogram_unet")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=localization_loss
)

print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Parameters:", model.count_params())
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Model ready.")

In [ ]:
BEST_MODEL_PATH = FOLDERS["models"] / "mammogram_unet_best.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        min_delta=1e-4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.TerminateOnNaN(),
]

history = model.fit(
    X_train,
    Y_train,
    validation_data=(X_val, Y_val),
    epochs=60,
    batch_size=8,
    shuffle=True,
    callbacks=callbacks,
    verbose=2
)

history_df = pd.DataFrame(history.history)
history_df.index.name = "epoch"
history_df.index += 1
history_df.to_csv(FOLDERS["results"] / "training_history.csv")

# Load the exact checkpoint selected by validation loss.
model = tf.keras.models.load_model(
    BEST_MODEL_PATH,
    compile=False
)

best_epoch = int(history_df["val_loss"].idxmin())

print("\nTraining complete.")
print("Epochs completed:", len(history_df))
print("Best epoch:", best_epoch)
print("Best validation loss:", history_df.loc[best_epoch, "val_loss"])
print("Saved model:", BEST_MODEL_PATH)

In [ ]:
val_scores = model.predict(X_val, batch_size=8, verbose=0)

assert val_scores.shape == Y_val.shape
assert np.isfinite(val_scores).all()

truth = Y_val[..., 0] > 0
positive = truth.any(axis=(1, 2))
normal = ~positive

threshold_results = []

for threshold in np.arange(0.05, 1.0, 0.05):
    predicted = val_scores[..., 0] >= threshold

    intersection = (truth & predicted).sum(axis=(1, 2))
    true_area = truth.sum(axis=(1, 2))
    predicted_area = predicted.sum(axis=(1, 2))
    union = true_area + predicted_area - intersection

    dice = (
        2 * intersection[positive]
        / (true_area[positive] + predicted_area[positive])
    )
    iou = intersection[positive] / union[positive]

    threshold_results.append({
        "threshold": float(threshold),
        "abnormal_mean_dice": float(dice.mean()),
        "abnormal_mean_iou": float(iou.mean()),
        "normal_images_with_any_highlight": int(
            (predicted_area[normal] > 0).sum()
        ),
        "normal_mean_highlighted_fraction": float(
            predicted[normal].mean()
        ),
    })

threshold_table = pd.DataFrame(threshold_results)

threshold_table.to_csv(
    FOLDERS["results"] / "validation_threshold_comparison.csv",
    index=False
)

print("Validation abnormal images:", int(positive.sum()))
print("Validation normal images:", int(normal.sum()))
print("Score range:", float(val_scores.min()), "to", float(val_scores.max()))

display(threshold_table.round(4))

In [ ]:
import json

best_row = threshold_table.loc[
    threshold_table["abnormal_mean_dice"].idxmax()
]
LOCALIZATION_THRESHOLD = float(best_row["threshold"])

config = {
    "image_size": IMAGE_SIZE,
    "threshold": LOCALIZATION_THRESHOLD,
    "threshold_selection": "Maximum validation abnormal-image mean Dice",
    "target": "Approximate MIAS abnormality circles, benign and malignant",
    "validation_mean_dice": float(best_row["abnormal_mean_dice"]),
    "validation_mean_iou": float(best_row["abnormal_mean_iou"]),
    "validation_normal_images_with_any_highlight": int(
        best_row["normal_images_with_any_highlight"]
    ),
    "validation_normal_images": int(normal.sum()),
}

with open(FOLDERS["models"] / "localization_config.json", "w") as file:
    json.dump(config, file, indent=2)

# First three abnormal and first two normal validation examples.
preview_indices = (
    np.flatnonzero(positive)[:3].tolist()
    + np.flatnonzero(normal)[:2].tolist()
)

fig, axes = plt.subplots(
    len(preview_indices), 3,
    figsize=(12, 4 * len(preview_indices))
)

for row_number, index in enumerate(preview_indices):
    image = X_val[index, ..., 0]
    target = Y_val[index, ..., 0]
    prediction = val_scores[index, ..., 0] >= LOCALIZATION_THRESHOLD
    image_id = val_rows.iloc[index]["image_id"]

    for ax in axes[row_number]:
        ax.imshow(image, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")

    axes[row_number, 0].set_title(f"{image_id}: mammogram")

    axes[row_number, 1].imshow(
        np.ma.masked_where(target == 0, target),
        cmap="autumn", alpha=0.5, vmin=0, vmax=1
    )
    axes[row_number, 1].set_title("Dataset annotation")

    axes[row_number, 2].imshow(
        np.ma.masked_where(~prediction, prediction.astype(float)),
        cmap="autumn", alpha=0.5, vmin=0, vmax=1
    )
    axes[row_number, 2].set_title("Model highlight")

plt.tight_layout()
fig.savefig(
    FOLDERS["results"] / "validation_prediction_preview.png",
    dpi=150, bbox_inches="tight"
)
plt.show()

print("Saved threshold:", round(LOCALIZATION_THRESHOLD, 2))
print("Highlights indicate predicted abnormal regions, not cancer diagnoses.")

In [ ]:
X_test, Y_test, test_rows = prepare_split("test")
test_scores = model.predict(X_test, batch_size=8, verbose=0)

assert test_scores.shape == Y_test.shape
assert np.isfinite(test_scores).all()

test_truth = Y_test[..., 0] > 0
test_pred = test_scores[..., 0] >= LOCALIZATION_THRESHOLD

axes = (1, 2)
true_area = test_truth.sum(axis=axes)
predicted_area = test_pred.sum(axis=axes)
intersection = (test_truth & test_pred).sum(axis=axes)
union = true_area + predicted_area - intersection

test_positive = true_area > 0
test_normal = ~test_positive

# Leave overlap metrics undefined for normal images.
dice = np.full(len(test_rows), np.nan)
iou = np.full(len(test_rows), np.nan)

dice[test_positive] = (
    2 * intersection[test_positive]
    / (true_area[test_positive] + predicted_area[test_positive])
)
iou[test_positive] = (
    intersection[test_positive] / union[test_positive]
)

test_results = test_rows[
    ["image_id", "patient_id", "label"]
].copy()

test_results["dice"] = dice
test_results["iou"] = iou
test_results["highlighted_pixels"] = predicted_area
test_results["highlighted_fraction"] = (
    predicted_area / (IMAGE_SIZE * IMAGE_SIZE)
)

test_results.to_csv(
    FOLDERS["results"] / "test_per_image_results.csv",
    index=False
)

test_metrics = {
    "threshold": LOCALIZATION_THRESHOLD,
    "test_patients": int(test_rows["patient_id"].nunique()),
    "test_images": len(test_rows),
    "abnormal_images": int(test_positive.sum()),
    "normal_images": int(test_normal.sum()),
    "abnormal_mean_dice": float(dice[test_positive].mean()),
    "abnormal_mean_iou": float(iou[test_positive].mean()),
    "abnormal_images_with_zero_overlap": int(
        (intersection[test_positive] == 0).sum()
    ),
    "normal_images_with_any_highlight": int(
        (predicted_area[test_normal] > 0).sum()
    ),
    "normal_mean_highlighted_fraction": float(
        test_pred[test_normal].mean()
    ),
}

with open(FOLDERS["results"] / "test_metrics.json", "w") as file:
    json.dump(test_metrics, file, indent=2)

for name, value in test_metrics.items():
    print(f"{name}: {value:.4f}" if isinstance(value, float)
          else f"{name}: {value}")

In [ ]:
reloaded_model = tf.keras.models.load_model(
    BEST_MODEL_PATH,
    compile=False
)

with open(FOLDERS["models"] / "localization_config.json") as file:
    saved_config = json.load(file)

# Check that saving/loading preserved every model weight.
for original, restored in zip(
    model.get_weights(),
    reloaded_model.get_weights()
):
    np.testing.assert_array_equal(original, restored)

# Repeat inference with the same data and batch size.
reloaded_scores = reloaded_model.predict(
    X_test,
    batch_size=8,
    verbose=0
)

max_difference = float(
    np.max(np.abs(reloaded_scores - test_scores))
)

changed_pixels = int(np.count_nonzero(
    (reloaded_scores >= saved_config["threshold"])
    != (test_scores >= LOCALIZATION_THRESHOLD)
))

print("Model weights match exactly.")
print("Maximum prediction difference:", max_difference)
print("Pixels whose thresholded result changed:", changed_pixels)
print("Saved threshold:", saved_config["threshold"])

np.testing.assert_allclose(
    reloaded_scores,
    test_scores,
    rtol=1e-4,
    atol=1e-5
)

assert np.isclose(
    saved_config["threshold"],
    LOCALIZATION_THRESHOLD
)

print("Prediction consistency check passed.")

In [ ]:
import zipfile
from google.colab import files

EXPORT_PATH = Path("/content/module3_mammogram_export.zip")

export_files = {
    BEST_MODEL_PATH: "models/mammogram_unet_best.keras",
    FOLDERS["models"] / "localization_config.json":
        "models/localization_config.json",
    FOLDERS["results"] / "test_metrics.json":
        "results/test_metrics.json",
    FOLDERS["results"] / "test_per_image_results.csv":
        "results/test_per_image_results.csv",
}

for path in export_files:
    if not path.is_file():
        raise FileNotFoundError(path)

with zipfile.ZipFile(
    EXPORT_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for source, destination in export_files.items():
        archive.write(source, arcname=destination)

print("Export ready:", EXPORT_PATH.name)
files.download(str(EXPORT_PATH))

In [ ]:
from google.colab import files

sample_id = test_rows.iloc[0]["image_id"]
print("Test image:", sample_id)

files.download(str(MIAS_DIR / f"{sample_id}.pgm"))

In [ ]:
sample_id = test_rows.iloc[0]["image_id"]

print("Test image:", sample_id)
display(
    test_results.loc[
        test_results["image_id"] == sample_id,
        ["image_id", "label", "dice", "iou", "highlighted_fraction"]
    ]
)

print(
    "Colab highlighted area:",
    f"{test_pred[0].mean():.2%}"
)